In [1]:
import pandas as pd
import re
import numpy as np

# Load data
df = pd.read_csv('data/filtered.csv')

# --- STEP 1: ROBUST WEIGHT CALCULATION ---

# Function to standardize everything to Grams (g) or Milliliters (ml)
# Assumption: 'kg' and 'l' are the dominant units in 'prices_unit_(£)'
def standardize_weight(row):
    price = row['prices_(£)']
    unit_price = row['prices_unit_(£)']
    price_unit = str(row['unit']).lower()
    
    # Method A: Calculation (Primary Source of Truth)
    # Price / Unit_Price gives us the weight in the unit specified
    if pd.notnull(price) and pd.notnull(unit_price) and unit_price > 0:
        calc_amt = price / unit_price
        
        # Convert to g/ml
        if price_unit in ['kg', 'l', 'litre']:
            return round(calc_amt * 1000, 0) # 0.4kg -> 400g
        elif price_unit in ['g', 'ml']:
            return round(calc_amt, 0)
        elif price_unit == '100g': # Sometimes price is per 100g
            return round(calc_amt * 100, 0)
        elif price_unit == '100ml':
            return round(calc_amt * 100, 0)
            
    return np.nan

df['clean_weight_g_ml'] = df.apply(standardize_weight, axis=1)

# Method B: Regex Extraction (Fallback)
# Extract '400g', '1.5kg', '500ml', '1l' from title
def extract_weight_from_title(text):
    if not isinstance(text, str): return np.nan
    text = text.lower()
    
    # Pattern: number + optional space + unit
    # We look for g, kg, ml, l, cl
    pattern = r'(\d+(\.\d+)?)\s?(kg|g|ml|l|cl|ltr)\b'
    match = re.search(pattern, text)
    
    if match:
        val = float(match.group(1))
        unit = match.group(3)
        
        if unit == 'kg': return val * 1000
        if unit == 'l' or unit == 'ltr': return val * 1000
        if unit == 'cl': return val * 10
        return val # already g or ml
        
    return np.nan

# Fill missing calculated weights with extracted weights
df['clean_weight_g_ml'] = df['clean_weight_g_ml'].fillna(df['names'].apply(extract_weight_from_title))

# --- STEP 2: NAME SANITIZATION ---

stop_words = [
    'Tesco', 'Sainsbury\'s', 'Sainsburys', 'Sains', 'ASDA', 'Morrisons', 
    'Essential', 'Value', 'Basics', 'Savers', 'Just Essentials', 'The Best', 
    'Taste the Difference', 'Extra Special', 'Finest', 'by Sainsbury\'s',
    'Grower\'s Harvest', 'Stamford Street', 'Stockwell', 'Creamfields' # Common fake farm brands
]

def clean_product_name(text):
    if not isinstance(text, str): return ""
    
    clean_text = text
    # Remove stop words (case insensitive)
    for word in stop_words:
        pattern = re.compile(re.escape(word), re.IGNORECASE)
        clean_text = pattern.sub("", clean_text)
    
    # Remove weights from name (since we have them in a separate column now)
    # This helps cluster "Rice 1kg" and "Rice 500g" into "Rice" if we wanted, 
    # but for now we keep it to ensure "Chopped Tomatoes" doesn't just become "Tomatoes" 
    # Actually, removing weight from title is good for pure semantic matching
    # pattern_weight = r'(\d+(\.\d+)?)\s?(kg|g|ml|l|cl|pack|x\d+)\b'
    # clean_text = re.sub(pattern_weight, "", clean_text, flags=re.IGNORECASE)
    
    # Remove special chars and extra spaces
    clean_text = re.sub(r'[^\w\s]', '', clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip().lower()
    
    return clean_text

df['clean_name'] = df['names'].apply(clean_product_name)

# --- STEP 3: ANALYSE RESULTS ---

print("Total Rows:", len(df))
print("Rows with Valid Weight:", df['clean_weight_g_ml'].notnull().sum())
print("Rows Missing Weight:", df['clean_weight_g_ml'].isnull().sum())

print("\n--- Sample Before vs After (Text Cleaning) ---")
print(df[['names', 'clean_name', 'clean_weight_g_ml']].sample(10))

# Check for duplicates after cleaning (Potential Categories!)
# Group by Clean Name + Weight
duplicates = df.groupby(['clean_name', 'clean_weight_g_ml']).size().reset_index(name='count')
print("\n--- Potential Categories Found (Count > 1) ---")
print(duplicates[duplicates['count'] > 1].sort_values('count', ascending=False).head(10))

# Save the phase 1 output
df.to_csv('phase1_cleaned.csv', index=False)

Total Rows: 65473
Rows with Valid Weight: 59717
Rows Missing Weight: 5756

--- Sample Before vs After (Text Cleaning) ---
                                                   names  \
33999                       Knorr Stock Cubes Lamb 8x10g   
24783                                   Morrisons Shiraz   
1911   Chewy Vites Real Fruit Juice Kids Multi-Vit + ...   
35294                     Sainsbury's Vanilla Icing 400g   
39852       Birra Moretti Zero Alcohol-Free Beer 4x330ml   
6921                Weetabix Crispy Minis Chocolate Chip   
64929                Tesco Lemon & Strawberry Crunch 70g   
2485                                ASDA Golden Marzipan   
59608                   Urban Fruit Snack Pack Mango 35G   
31980             Sainsbury's Prunes in Fruit Juice 410g   

                                              clean_name  clean_weight_g_ml  
33999                       knorr stock cubes lamb 8x10g               80.0  
24783                                             shiraz     

In [5]:
df[df['clean_weight_g_ml'].isnull()].sample(10)

,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand,clean_weight_g_ml,clean_name
28851,Morrisons,1.69,0.56,unit,Morrisons Savers Salad Peppers,20240413,fresh_food,True,NaN,salad peppers
10663,ASDA,2.00,0.50,unit,The BAKERY at ASDA 4 Sweet Classic Battenberg ...,20240413,bakery,True,NaN,the bakery at 4 sweet classic battenberg cupcakes
10784,ASDA,2.75,0.46,unit,ASDA 6 Double Chocolate Muffins,20240413,bakery,True,NaN,6 double chocolate muffins
48390,Sains,8.00,8.00,unit,Sainsbury's Chrysanthemum Bowl Wrap 21cm,20240309,fresh_food,True,NaN,chrysanthemum bowl wrap 21cm
26008,Morrisons,6.49,0.33,unit,Velo Ice Cool Nicotine Pouches 10mg,20240413,drinks,False,NaN,velo ice cool nicotine pouches 10mg
774,ASDA,3.00,3.00,unit,George Home Disney Winnie The Pooh Mug,20240413,drinks,True,NaN,george home disney winnie the pooh mug
11154,ASDA,1.00,1.00,unit,George Home How Old? Large Candle,20240413,bakery,True,NaN,george home how old large candle
31318,Morrisons,2.00,0.20,unit,Morrisons Mini Brownies,20240127,bakery,True,NaN,mini brownies
10724,ASDA,1.50,0.75,unit,The BAKERY at ASDA Chocolate Flavour Crodoughs...,20240413,bakery,True,NaN,the bakery at chocolate flavour crodoughs 2pk
64708,Tesco,1.50,0.50,unit,Tesco Persimmons Minimum 3 Pack,20240303,fresh_food,True,NaN,persimmons minimum 3 pack
